# coord_verification for ps2

In [1]:

import numpy as np
import torch
from pathlib import Path
import json
import struct
from PIL import Image
import os

# ============================================================================
# Part 1: Calibration Data Generator (同じ)
# ============================================================================

class CalibrationDataGenerator:
    """既知の3D座標を持つ合成シーンデータの生成"""
    
    def __init__(self, num_views=4, num_points=100):
        self.num_views = num_views
        self.num_points = num_points
        
    def generate_cube_points(self, size=1.0):
        """立方体形状の3D点群を生成"""
        points = []
        
        # 立方体の8頂点
        for x in [-size, size]:
            for y in [-size, size]:
                for z in [-size, size]:
                    points.append([x, y, z])
        
        # 辺上の点
        n_edge = max(1, (self.num_points - 8) // 12)
        for i in range(n_edge):
            t = (i + 1) / (n_edge + 1)
            points.extend([
                [2*size*t - size, -size, -size],
                [2*size*t - size, size, -size],
                [2*size*t - size, -size, size],
                [2*size*t - size, size, size],
                [-size, 2*size*t - size, -size],
                [size, 2*size*t - size, -size],
                [-size, 2*size*t - size, size],
                [size, 2*size*t - size, size],
                [-size, -size, 2*size*t - size],
                [size, -size, 2*size*t - size],
                [-size, size, 2*size*t - size],
                [size, size, 2*size*t - size],
            ])
        
        return np.array(points[:self.num_points])
    
    def generate_circular_camera_poses(self, radius=5.0, height=0.0):
        """円形配置のカメラポーズを生成"""
        poses = []
        for i in range(self.num_views):
            angle = 2 * np.pi * i / self.num_views
            
            cam_pos = np.array([
                radius * np.cos(angle),
                radius * np.sin(angle),
                height
            ])
            
            forward = -cam_pos / np.linalg.norm(cam_pos)
            up = np.array([0, 0, 1])
            right = np.cross(up, forward)
            right = right / np.linalg.norm(right)
            up = np.cross(forward, right)
            
            R = np.stack([right, up, forward], axis=0)
            t = -R @ cam_pos
            
            pose = np.eye(4)
            pose[:3, :3] = R
            pose[:3, 3] = t
            
            poses.append(pose)
            
        return poses
    
    def create_mock_scene_for_process2(self, device='cpu'):
        """Process2用のMASt3Rシーンオブジェクトを作成"""
        
        pts3d_world = self.generate_cube_points(size=1.0)
        camera_poses = self.generate_circular_camera_poses(radius=5.0, height=0.5)
        
        class MockSceneProcess2:
            def __init__(self, pts3d_world, camera_poses, device):
                self.device = device
                self.num_views = len(camera_poses)
                self.num_points = len(pts3d_world)
                
                # Ground truthを保存
                self.ground_truth_pts3d_world = pts3d_world.copy()
                self.ground_truth_poses = [p.copy() for p in camera_poses]
                
                # Process2スタイル: im_pts3d, im_conf属性を持つ
                self.im_pts3d = []
                self.im_conf = []
                
                for pose in camera_poses:
                    # カメラ座標系の点
                    pts_cam = self._world_to_camera(pts3d_world, pose)
                    self.im_pts3d.append(torch.from_numpy(pts_cam).float().to(device))
                    # 高い信頼度
                    self.im_conf.append(torch.ones(self.num_points).float().to(device) * 3.0)
                
                # カメラ内部パラメータ
                self.im_focals = torch.tensor([[500.0]] * self.num_views).to(device)
                self.im_pp = torch.tensor([[112.0, 112.0]] * self.num_views).to(device)  # MASt3Rサイズ基準
                
                # Camera-to-world poses (C2W)
                self.im_poses = torch.stack([
                    torch.eye(4) for _ in range(self.num_views)
                ]).float().to(device)
                
                for i, pose in enumerate(camera_poses):
                    # W2CをC2Wに変換
                    c2w = torch.from_numpy(np.linalg.inv(pose)).float()
                    self.im_poses[i] = c2w
            
            def _world_to_camera(self, pts_world, pose):
                """ワールド座標からカメラ座標へ変換"""
                pts_homo = np.hstack([pts_world, np.ones((len(pts_world), 1))])
                pts_cam_homo = (pose @ pts_homo.T).T
                return pts_cam_homo[:, :3]
            
            def get_im_poses(self):
                return self.im_poses
            
            def get_focals(self):
                return self.im_focals
            
            def get_principal_points(self):
                return self.im_pp
        
        scene = MockSceneProcess2(pts3d_world, camera_poses, device)
        return scene
    
    def save_ground_truth(self, scene, output_path):
        """Ground truthデータを保存"""
        output_path = Path(output_path)
        output_path.mkdir(parents=True, exist_ok=True)
        
        np.savetxt(
            output_path / 'ground_truth_points3d.txt',
            scene.ground_truth_pts3d_world,
            header='X Y Z (world coordinates)',
            fmt='%.6f'
        )
        
        with open(output_path / 'ground_truth_poses.json', 'w') as f:
            poses_list = [p.tolist() for p in scene.ground_truth_poses]
            json.dump({
                'poses': poses_list,
                'description': '4x4 transformation matrices (world to camera)'
            }, f, indent=2)
        
        print(f"✓ Ground truth saved to {output_path}")

# process2

In [2]:

def rotmat_to_qvec(R):
    """回転行列をクォータニオンに変換"""
    R = np.asarray(R, dtype=np.float64)
    trace = np.trace(R)

    if trace > 0:
        s = 0.5 / np.sqrt(trace + 1.0)
        w = 0.25 / s
        x = (R[2, 1] - R[1, 2]) * s
        y = (R[0, 2] - R[2, 0]) * s
        z = (R[1, 0] - R[0, 1]) * s
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        s = 2.0 * np.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2])
        w = (R[2, 1] - R[1, 2]) / s
        x = 0.25 * s
        y = (R[0, 1] + R[1, 0]) / s
        z = (R[0, 2] + R[2, 0]) / s
    elif R[1, 1] > R[2, 2]:
        s = 2.0 * np.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2])
        w = (R[0, 2] - R[2, 0]) / s
        x = (R[0, 1] + R[1, 0]) / s
        y = 0.25 * s
        z = (R[1, 2] + R[2, 1]) / s
    else:
        s = 2.0 * np.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1])
        w = (R[1, 0] - R[0, 1]) / s
        x = (R[0, 2] + R[2, 0]) / s
        y = (R[1, 2] + R[2, 1]) / s
        z = 0.25 * s

    qvec = np.array([w, x, y, z], dtype=np.float64)
    qvec = qvec / np.linalg.norm(qvec)

    return qvec


def write_cameras_binary(cameras_dict, image_size, output_file):
    """
    cameras.binを出力（PINHOLEモデル使用）
    """
    width, height = image_size
    num_cameras = len(cameras_dict)

    # COLMAP camera models
    PINHOLE = 1  # 🔧 SIMPLE_PINHOLE (0) から PINHOLE (1) に変更

    with open(output_file, 'wb') as f:
        f.write(struct.pack('Q', num_cameras))

        for camera_id, (img_id, cam_params) in enumerate(cameras_dict.items(), start=1):
            focal = cam_params['focal']

            # PINHOLEの場合: fx, fy, cx, cy
            #fx = fy = focal  # 等方性カメラを仮定
            
            #new settiing 2026/01/26
            if isinstance(focal, (tuple, list)):
                fx, fy = focal
            else:
                fx = fy = focal
            

            # Principal pointを取得（存在しない場合は中心）
            if 'pp' in cam_params:
                pp = cam_params['pp']
                cx = float(pp[0])
                cy = float(pp[1])
            else:
                cx = width / 2.0
                cy = height / 2.0

            # camera_id
            f.write(struct.pack('I', camera_id))
            # model_id (PINHOLE = 1)
            f.write(struct.pack('i', PINHOLE))
            # width
            f.write(struct.pack('Q', width))
            # height
            f.write(struct.pack('Q', height))
            # params: fx, fy, cx, cy (4パラメータ)
            f.write(struct.pack('d', fx))
            f.write(struct.pack('d', fy))
            f.write(struct.pack('d', cx))
            f.write(struct.pack('d', cy))

    print(f"COLMAP cameras.bin saved to {output_file}")


def write_images_binary(cameras_dict, output_file):
    """images.binを出力"""
    num_images = len(cameras_dict)

    with open(output_file, 'wb') as f:
        f.write(struct.pack('Q', num_images))

        for image_id, (img_id, cam_params) in enumerate(cameras_dict.items(), start=1):
            R = cam_params['rotation']
            quat = rotmat_to_qvec(R)
            t = cam_params['translation']
            camera_id = image_id

            f.write(struct.pack('I', image_id))
            for q in quat:
                f.write(struct.pack('d', q))
            for ti in t:
                f.write(struct.pack('d', ti))
            f.write(struct.pack('I', camera_id))

            name_bytes = img_id.encode('utf-8') + b'\x00'
            f.write(name_bytes)
            f.write(struct.pack('Q', 0))

    print(f"COLMAP images.bin saved to {output_file}")


def write_points3D_binary(pts3d, confidence, output_file):
    """points3D.binを出力"""
    num_points = len(pts3d)

    with open(output_file, 'wb') as f:
        f.write(struct.pack('Q', num_points))

        for point_id, pt in enumerate(pts3d, start=1):
            x, y, z = pt

            f.write(struct.pack('Q', point_id))
            f.write(struct.pack('d', x))
            f.write(struct.pack('d', y))
            f.write(struct.pack('d', z))

            # RGB (グレー)
            f.write(struct.pack('B', 128))
            f.write(struct.pack('B', 128))
            f.write(struct.pack('B', 128))

            # error
            if confidence is not None and point_id <= len(confidence):
                error = 1.0 / max(confidence[point_id-1], 0.001)
            else:
                error = 1.0
            f.write(struct.pack('d', error))

            # track_length
            f.write(struct.pack('Q', 0))

    print(f"COLMAP points3D.bin saved to {output_file}")


def export_colmap_binary(cameras_dict, pts3d, confidence, image_size, output_dir):
    """COLMAPバイナリファイルを出力"""
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    write_cameras_binary(
        cameras_dict,
        image_size,
        output_path / 'cameras.bin'
    )

    write_images_binary(
        cameras_dict,
        output_path / 'images.bin'
    )

    write_points3D_binary(
        pts3d,
        confidence,
        output_path / 'points3D.bin'
    )

    print(f"\nCOLMAP binary files exported to {output_dir}/")
    print(f"  - cameras.bin: {len(cameras_dict)} cameras (PINHOLE model)")
    print(f"  - images.bin: {len(cameras_dict)} images")
    print(f"  - points3D.bin: {len(pts3d)} points")


# =====================================================================
# CELL 11: Camera Parameter Extraction (REVISED 2026/01/26)
# =====================================================================
def extract_camera_params_process2(scene, image_paths, conf_threshold=1.5):
    """
    Extracts camera parameters and 3D points from the scene (FIXED: proper fx, fy handling).
    """
    print("\n=== Extracting Camera Parameters ===")

    cameras_dict = {}
    all_pts3d = []
    all_confidence = []

    try:
        # Attempt to get camera poses
        if hasattr(scene, 'get_im_poses'):
            poses = scene.get_im_poses()
        elif hasattr(scene, 'im_poses'):
            poses = scene.im_poses
        else:
            poses = None

        # Attempt to get focal lengths
        if hasattr(scene, 'get_focals'):
            focals = scene.get_focals()
        elif hasattr(scene, 'im_focals'):
            focals = scene.im_focals
        else:
            focals = None

        # Attempt to get principal points
        if hasattr(scene, 'get_principal_points'):
            pps = scene.get_principal_points()
        elif hasattr(scene, 'im_pp'):
            pps = scene.im_pp
        else:
            pps = None
    except Exception as e:
        print(f"⚠️ Error getting camera parameters: {e}")
        poses = None
        focals = None
        pps = None

    # [Important] MASt3R internal processing size
    mast3r_size = 224.0

    n_images = min(len(poses) if poses is not None else len(image_paths), len(image_paths))

    for idx in range(n_images):
        img_name = os.path.basename(image_paths[idx])

        try:
            # Get original image dimensions
            img = Image.open(image_paths[idx])
            W, H = img.size
            img.close()

            # Calculate scaling ratio
            scale = W / mast3r_size

            # Get Pose (Convert camera-to-world to world-to-camera)
            if poses is not None and idx < len(poses):
                pose_c2w = poses[idx]
                if isinstance(pose_c2w, torch.Tensor):
                    pose_c2w = pose_c2w.detach().cpu().numpy()
                if not isinstance(pose_c2w, np.ndarray) or pose_c2w.shape != (4, 4):
                    pose_c2w = np.eye(4)

                # Invert to get world-to-camera pose
                pose = np.linalg.inv(pose_c2w)
            else:
                pose = np.eye(4)

            # 🔧 FIX: Get and scale focal length (handle both isotropic and anisotropic)
            if focals is not None and idx < len(focals):
                focal_mast3r = focals[idx]
                if isinstance(focal_mast3r, torch.Tensor):
                    focal_mast3r = focal_mast3r.detach().cpu()

                # Check if isotropic (fx = fy) or anisotropic (fx ≠ fy)
                if focals.shape[1] == 1:
                    # Isotropic camera (fx = fy)
                    focal_val = float(focal_mast3r) if focal_mast3r.numel() == 1 else float(focal_mast3r[0])
                    fx = fy = focal_val * scale
                else:
                    # Anisotropic camera (fx ≠ fy)
                    fx = float(focal_mast3r[0]) * scale
                    fy = float(focal_mast3r[1]) * scale
            else:
                # Default fallback
                fx = fy = 1000.0

            # Get and scale principal point
            if pps is not None and idx < len(pps):
                pp_mast3r = pps[idx]
                if isinstance(pp_mast3r, torch.Tensor):
                    pp_mast3r = pp_mast3r.detach().cpu().numpy()

                # 🔧 Apply scaling
                pp = pp_mast3r * scale
            else:
                pp = np.array([W / 2.0, H / 2.0])

            # 🔧 FIX: Store camera parameters with focal as tuple (fx, fy)
            cameras_dict[img_name] = {
                'focal': (fx, fy),  # ← FIXED: Store as tuple
                'pp': pp,
                'pose': pose,
                'rotation': pose[:3, :3],
                'translation': pose[:3, 3],
                'width': W,
                'height': H
            }

            # Debugging info (First image only)
            if idx == 0:
                print(f"\nExample camera 0:")
                print(f"  Original size: {W}x{H}")
                print(f"  MASt3R size: {mast3r_size}")
                print(f"  Scale factor: {scale:.3f}")
                print(f"  focals.shape: {focals.shape}")
                if focals.shape[1] == 1:
                    print(f"  MASt3R focal: {focal_val:.2f}")
                    print(f"  Scaled focal: fx = fy = {fx:.2f}")
                else:
                    print(f"  MASt3R focals: fx={float(focal_mast3r[0]):.2f}, fy={float(focal_mast3r[1]):.2f}")
                    print(f"  Scaled focals: fx={fx:.2f}, fy={fy:.2f}")
                print(f"  MASt3R pp: [{pp_mast3r[0]:.2f}, {pp_mast3r[1]:.2f}]")
                print(f"  Scaled pp: [{pp[0]:.2f}, {pp[1]:.2f}]")

            # Extract 3D points
            if hasattr(scene, 'im_pts3d') and idx < len(scene.im_pts3d):
                pts3d_img = scene.im_pts3d[idx]
            elif hasattr(scene, 'get_pts3d'):
                pts3d_all = scene.get_pts3d()
                pts3d_img = pts3d_all[idx] if idx < len(pts3d_all) else None
            else:
                pts3d_img = None

            # Extract confidence scores
            if hasattr(scene, 'im_conf') and idx < len(scene.im_conf):
                conf_img = scene.im_conf[idx]
            elif hasattr(scene, 'get_conf'):
                conf_all = scene.get_conf()
                conf_img = conf_all[idx] if idx < len(conf_all) else None
            else:
                conf_img = None

            # Process 3D points and confidence
            if pts3d_img is not None:
                if isinstance(pts3d_img, torch.Tensor):
                    pts3d_img = pts3d_img.detach().cpu().numpy()

                pts3d_flat = pts3d_img.reshape(-1, 3) if pts3d_img.ndim == 3 else pts3d_img
                all_pts3d.append(pts3d_flat)

                if conf_img is not None:
                    if isinstance(conf_img, (list, torch.Tensor)):
                        conf_img = np.array(conf_img) if isinstance(conf_img, list) else conf_img.detach().cpu().numpy()

                    conf_flat = conf_img.reshape(-1) if conf_img.ndim > 1 else conf_img
                    
                    if len(conf_flat) != len(pts3d_flat):
                        conf_flat = np.ones(len(pts3d_flat))
                    
                    all_confidence.append(conf_flat)
                else:
                    all_confidence.append(np.ones(len(pts3d_flat)))

        except Exception as e:
            print(f"⚠️ Error processing image {idx} ({img_name}): {e}")
            # Fallback to default values with scaling applied
            img = Image.open(image_paths[idx])
            W, H = img.size
            img.close()

            cameras_dict[img_name] = {
                'focal': (1000.0 * (W / mast3r_size), 1000.0 * (W / mast3r_size)),  # ← FIXED: Tuple
                'pp': np.array([W / 2.0, H / 2.0]),
                'pose': np.eye(4),
                'rotation': np.eye(3),
                'translation': np.zeros(3),
                'width': W,
                'height': H
            }
            continue

    # Consolidate all 3D points
    if all_pts3d:
        pts3d = np.vstack(all_pts3d)
        confidence = np.concatenate(all_confidence)
    else:
        pts3d = np.zeros((0, 3))
        confidence = np.zeros(0)

    print(f"✓ Extracted parameters for {len(cameras_dict)} cameras")
    print(f"✓ Total 3D points: {len(pts3d)}")

    # Filter points by confidence
    if len(confidence) > 0:
        valid_mask = confidence > conf_threshold
        pts3d = pts3d[valid_mask]
        confidence = confidence[valid_mask]
        print(f"✓ Points after confidence filtering (>{conf_threshold}): {len(pts3d)}")

    return cameras_dict, pts3d, confidence

# =====================================================================
# Complete Color Extraction for Process2 (newly defined 2026/01/26)
# =====================================================================

import numpy as np
from PIL import Image
import struct
from pathlib import Path

# =====================================================================
# STEP 1: Color Extraction Function
# =====================================================================

def extract_colors_from_images(scene, image_paths, pts3d, confidence, conf_threshold=1.5):
    """
    Extract colors from images that match the filtered pts3d.
    
    This matches Traditional method's color extraction.
    
    Args:
        scene: MASt3R scene object
        image_paths: List of image file paths
        pts3d: (N, 3) filtered 3D points (after confidence filtering)
        confidence: (N,) filtered confidence scores
        conf_threshold: Confidence threshold used for filtering
    
    Returns:
        colors: (N, 3) RGB colors [0-255] matching pts3d
    """
    print("\n=== Extracting Colors from Images ===")
    
    # Get all 3D points BEFORE filtering (to match with colors)
    all_pts3d = []
    for idx in range(len(image_paths)):
        if hasattr(scene, 'im_pts3d') and idx < len(scene.im_pts3d):
            pts3d_img = scene.im_pts3d[idx]
        elif hasattr(scene, 'get_pts3d'):
            pts3d_all = scene.get_pts3d()
            pts3d_img = pts3d_all[idx] if idx < len(pts3d_all) else None
        else:
            pts3d_img = None
        
        if pts3d_img is not None:
            if isinstance(pts3d_img, torch.Tensor):
                pts3d_img = pts3d_img.detach().cpu().numpy()
            pts3d_flat = pts3d_img.reshape(-1, 3) if pts3d_img.ndim == 3 else pts3d_img
            all_pts3d.append(pts3d_flat)
    
    # Get dimensions from first image
    first_img = Image.open(image_paths[0])
    W_orig, H_orig = first_img.size
    first_img.close()
    
    # MASt3R uses 224x224 internally
    mast3r_size = 224
    
    # Extract colors from all images
    print(f"Extracting colors from {len(image_paths)} images...")
    all_colors = []
    
    for idx, img_path in enumerate(image_paths):
        # Open and resize image to MASt3R size (224x224)
        img = Image.open(img_path)
        img_resized = img.resize((mast3r_size, mast3r_size), Image.BILINEAR)
        img_array = np.array(img_resized)  # Shape: (224, 224, 3)
        img.close()
        
        # Reshape to (224*224, 3) to match point order
        colors_flat = img_array.reshape(-1, 3)
        all_colors.append(colors_flat)
        
        if idx == 0:
            print(f"  Example image 0:")
            print(f"    Original size: {W_orig}x{H_orig}")
            print(f"    Resized to: {mast3r_size}x{mast3r_size}")
            print(f"    Colors shape: {colors_flat.shape}")
    
    # Stack all colors
    colors_all = np.vstack(all_colors)  # Shape: (N_total, 3)
    print(f"✓ Total colors extracted: {len(colors_all):,}")
    
    # Get confidence for all points (before filtering)
    all_conf = []
    for idx in range(len(image_paths)):
        if hasattr(scene, 'im_conf') and idx < len(scene.im_conf):
            conf_img = scene.im_conf[idx]
        elif hasattr(scene, 'get_conf'):
            conf_all = scene.get_conf()
            conf_img = conf_all[idx] if idx < len(conf_all) else None
        else:
            conf_img = None
        
        if conf_img is not None:
            if isinstance(conf_img, torch.Tensor):
                conf_img = conf_img.detach().cpu().numpy()
            conf_flat = conf_img.reshape(-1) if conf_img.ndim > 1 else conf_img
        else:
            conf_flat = np.ones(len(all_pts3d[idx]))
        
        all_conf.append(conf_flat)
    
    conf_all = np.concatenate(all_conf)
    
    # Apply THE SAME filtering as pts3d
    valid_mask = conf_all > conf_threshold
    colors_filtered = colors_all[valid_mask]
    
    print(f"✓ Colors after confidence filtering (>{conf_threshold}): {len(colors_filtered):,}")
    
    # Verify shapes match
    if len(colors_filtered) != len(pts3d):
        print(f"⚠️ WARNING: Color count ({len(colors_filtered)}) != Point count ({len(pts3d)})")
        print(f"  Adjusting to match...")
        min_len = min(len(colors_filtered), len(pts3d))
        colors_filtered = colors_filtered[:min_len]
    else:
        print(f"✓ Colors match points: {len(colors_filtered):,} colors for {len(pts3d):,} points")
    
    # Verify colors are diverse
    unique_colors = len(np.unique(colors_filtered, axis=0))
    print(f"✓ Unique colors: {unique_colors:,}")
    
    if unique_colors < 100:
        print(f"⚠️ WARNING: Very few unique colors!")
    else:
        print(f"✓ Good color diversity")
    
    return colors_filtered


# =====================================================================
# STEP 2: Write points3D.bin with Colors
# =====================================================================

def write_points3D_binary_with_colors(pts3d, confidence, colors, output_file):
    """
    Export points3D.bin with actual colors.
    
    Args:
        pts3d: (N, 3) array of 3D points
        confidence: (N,) array of confidence scores
        colors: (N, 3) array of RGB colors [0-255]
        output_file: Path to output file
    """
    num_points = len(pts3d)

    with open(output_file, 'wb') as f:
        f.write(struct.pack('Q', num_points))

        for point_id, (pt, color) in enumerate(zip(pts3d, colors), start=1):
            x, y, z = pt

            f.write(struct.pack('Q', point_id))
            f.write(struct.pack('d', x))
            f.write(struct.pack('d', y))
            f.write(struct.pack('d', z))

            # RGB Color (ACTUAL colors now!)
            r = int(np.clip(color[0], 0, 255))
            g = int(np.clip(color[1], 0, 255))
            b = int(np.clip(color[2], 0, 255))
            
            f.write(struct.pack('B', r))
            f.write(struct.pack('B', g))
            f.write(struct.pack('B', b))

            # Error estimation
            if confidence is not None and point_id <= len(confidence):
                error = 1.0 / max(confidence[point_id-1], 0.001)
            else:
                error = 1.0
            f.write(struct.pack('d', error))

            # track_length (Set to 0)
            f.write(struct.pack('Q', 0))

    print(f"COLMAP points3D.bin saved to {output_file}")
    print(f"  ✓ With actual RGB colors from images!")


# =====================================================================
# STEP 3: Export with Colors
# =====================================================================

def export_colmap_binary_with_colors(cameras_dict, pts3d, confidence, colors, 
                                     image_size, output_dir):
    """
    Export COLMAP binary files with actual colors.
    
    Args:
        cameras_dict: Dictionary of camera parameters
        pts3d: (N, 3) filtered 3D points
        confidence: (N,) filtered confidence scores
        colors: (N, 3) RGB colors [0-255]
        image_size: (width, height) tuple
        output_dir: Output directory path
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    # Write cameras.bin (same as before)
    write_cameras_binary(
        cameras_dict,
        image_size,
        output_path / 'cameras.bin'
    )

    # Write images.bin (same as before)
    write_images_binary(
        cameras_dict,
        output_path / 'images.bin'
    )

    # Write points3D.bin WITH COLORS (NEW!)
    write_points3D_binary_with_colors(
        pts3d,
        confidence,
        colors,  # ← Actual colors!
        output_path / 'points3D.bin'
    )

    print(f"\n✓ COLMAP binary files exported to {output_dir}/")
    print(f"  - cameras.bin: {len(cameras_dict)} cameras (PINHOLE model)")
    print(f"  - images.bin: {len(cameras_dict)} images")
    print(f"  - points3D.bin: {len(pts3d)} points WITH COLORS")


# =====================================================================
# STEP 4: Complete Workflow
# =====================================================================

def create_process2_with_colors(scene, image_paths, output_dir, conf_threshold=1.5):
    """
    Complete workflow: Process2 with color extraction.
    
    Usage:
        create_process2_with_colors(
            scene, 
            image_paths, 
            '/kaggle/working/output/sparse_process2_with_colors/0',
            conf_threshold=1.5
        )
    """
    print("="*80)
    print("CREATING PROCESS2 COLMAP WITH COLORS")
    print("="*80)
    
    # Step 1: Extract camera parameters and points
    cameras_dict, pts3d, confidence = extract_camera_params_process2(
        scene, image_paths, conf_threshold=conf_threshold
    )
    
    print(f"\n✓ Extracted:")
    print(f"  - {len(cameras_dict)} cameras")
    print(f"  - {len(pts3d):,} 3D points")
    
    # Step 2: Extract colors (NEW!)
    colors = extract_colors_from_images(
        scene, image_paths, pts3d, confidence, conf_threshold
    )
    
    # Step 3: Get image size
    img = Image.open(image_paths[0])
    image_size = img.size
    img.close()
    
    # Step 4: Export with colors
    export_colmap_binary_with_colors(
        cameras_dict, pts3d, confidence, colors,
        image_size, output_dir
    )
    
    print("\n" + "="*80)
    print("✓ COMPLETE!")
    print("="*80)
    print("\nOutput directory:", output_dir)
    print("\nNext steps:")
    print("1. Train 3DGS with this reconstruction")
    print("2. Compare quality with gray Process2 and Traditional")
    print("3. Check if colors improve geometry convergence")
    
    return cameras_dict, pts3d, confidence, colors

In [3]:
# ============================================================================
# Part 3: テキスト形式での保存 (検証用)
# ============================================================================

def save_colmap_text_process2(cameras_dict, pts3d, confidence, output_dir):
    """Process2: COLMAP形式をテキストで保存"""
    print("\n=== Saving COLMAP in text format for verification ===")
    
    text_dir = Path(output_dir) / 'text_format'
    text_dir.mkdir(parents=True, exist_ok=True)
    
    # cameras.txt
    with open(text_dir / 'cameras.txt', 'w') as f:
        f.write("# Camera list with one line of data per camera:\n")
        f.write("#   CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]\n")
        for camera_id, (img_name, cam) in enumerate(cameras_dict.items(), start=1):
            focal = cam['focal']
            fx, fy = focal if isinstance(focal, (tuple, list)) else (focal, focal)
            pp = cam['pp']
            f.write(f"{camera_id} PINHOLE {cam['width']} {cam['height']} ")
            f.write(f"{fx:.6f} {fy:.6f} {pp[0]:.6f} {pp[1]:.6f}\n")
    
    # images.txt
    with open(text_dir / 'images.txt', 'w') as f:
        f.write("# Image list with two lines of data per image:\n")
        f.write("#   IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME\n")
        f.write("#   POINTS2D[] as (X, Y, POINT3D_ID)\n")
        for image_id, (img_name, cam) in enumerate(cameras_dict.items(), start=1):
            R = cam['rotation']
            t = cam['translation']
            qvec = rotmat_to_qvec(R)
            
            f.write(f"{image_id} ")
            f.write(" ".join([f"{q:.6f}" for q in qvec]))
            f.write(" ")
            f.write(" ".join([f"{tv:.6f}" for tv in t]))
            f.write(f" {image_id} {img_name}\n")
            f.write("\n")
    
    # points3D.txt
    with open(text_dir / 'points3D.txt', 'w') as f:
        f.write("# 3D point list with one line of data per point:\n")
        f.write("#   POINT3D_ID, X, Y, Z, R, G, B, ERROR, TRACK[] as (IMAGE_ID, POINT2D_IDX)\n")
        
        for i, pt in enumerate(pts3d):
            f.write(f"{i} {pt[0]:.6f} {pt[1]:.6f} {pt[2]:.6f} 128 128 128 0.0\n")
    
    print(f"✓ Saved text format to {text_dir}")
    return text_dir


# ============================================================================
# Part 4: 検証関数
# ============================================================================

def verify_transformation(ground_truth_path, colmap_text_path):
    """COLMAP出力とGround truthを比較"""
    ground_truth_path = Path(ground_truth_path)
    colmap_text_path = Path(colmap_text_path)
    
    # Ground truth読み込み
    gt_points = np.loadtxt(ground_truth_path / 'ground_truth_points3d.txt')
    
    # COLMAP points3D.txt読み込み
    colmap_points_file = colmap_text_path / 'points3D.txt'
    if not colmap_points_file.exists():
        print(f"❌ COLMAP output not found: {colmap_points_file}")
        return False
    
    colmap_points = []
    with open(colmap_points_file, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.strip().split()
            if len(parts) >= 4:
                x, y, z = float(parts[1]), float(parts[2]), float(parts[3])
                colmap_points.append([x, y, z])
    
    colmap_points = np.array(colmap_points)
    
    print("\n" + "="*70)
    print("process2検証結果")
    print("="*70)
    print(f"\nGround truth点数: {len(gt_points)}")
    print(f"COLMAP点数: {len(colmap_points)}")
    
    if len(colmap_points) == 0:
        print("❌ COLMAP出力に点が見つかりません")
        return False
    
    print("\n【Ground Truth統計】")
    print(f"  平均: {gt_points.mean(axis=0)}")
    print(f"  標準偏差: {gt_points.std(axis=0)}")
    print(f"  最小値: {gt_points.min(axis=0)}")
    print(f"  最大値: {gt_points.max(axis=0)}")
    
    print("\n【COLMAP出力統計】")
    print(f"  平均: {colmap_points.mean(axis=0)}")
    print(f"  標準偏差: {colmap_points.std(axis=0)}")
    print(f"  最小値: {colmap_points.min(axis=0)}")
    print(f"  最大値: {colmap_points.max(axis=0)}")
    
    # 座標サンプル比較
    print("\n【座標サンプル比較 (最初の5点)】")
    print("Ground Truth:")
    for i in range(min(5, len(gt_points))):
        print(f"  {i}: [{gt_points[i][0]:8.4f}, {gt_points[i][1]:8.4f}, {gt_points[i][2]:8.4f}]")
    print("\nCOLMAP Output:")
    for i in range(min(5, len(colmap_points))):
        print(f"  {i}: [{colmap_points[i][0]:8.4f}, {colmap_points[i][1]:8.4f}, {colmap_points[i][2]:8.4f}]")
    
    # Procrustes解析
    if len(colmap_points) >= len(gt_points):
        try:
            from scipy.spatial import procrustes
            
            n_compare = min(len(gt_points), len(colmap_points))
            gt_compare = gt_points[:n_compare]
            colmap_compare = colmap_points[:n_compare]
            
            gt_centered = gt_compare - gt_compare.mean(axis=0)
            colmap_centered = colmap_compare - colmap_compare.mean(axis=0)
            
            mtx1, mtx2, disparity = procrustes(gt_centered, colmap_centered)
            
            print(f"\n【Procrustes解析】")
            print(f"  比較点数: {n_compare}")
            print(f"  差異度: {disparity:.6f}")
            
            if disparity < 0.01:
                print("  判定: ✓ 座標一致(剛体変換のみ)")
            elif disparity < 0.1:
                print("  判定: ⚠ 概ね一致(小さな差異あり)")
            else:
                print("  判定: ❌ 大きな座標変化を検出")
        except ImportError:
            print("\n⚠ scipy未インストールのため詳細解析をスキップ")
        except Exception as e:
            print(f"\n⚠ Procrustes解析エラー: {e}")
    
    print("="*70 + "\n")
    return True


# ============================================================================
# Part 5: 統合テスト関数
# ============================================================================

def test_process2_with_calibration(num_views=4, num_points=100, 
                                   output_dir='/kaggle/working/calibration_test_process2',
                                   conf_threshold=1.5):
    """Process2とキャリブレーションデータでEnd-to-Endテスト"""
    
    print("="*70)
    print("Process2 COLMAP Method + Calibration Test")
    print("="*70)
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # [1] キャリブレーションデータ生成
    print("\n[1/6] キャリブレーションデータ生成中...")
    generator = CalibrationDataGenerator(num_views=num_views, num_points=num_points)
    scene = generator.create_mock_scene_for_process2(device='cpu')
    
    ground_truth_dir = output_dir / 'ground_truth'
    generator.save_ground_truth(scene, ground_truth_dir)
    
    print(f"  ビュー数: {scene.num_views}")
    print(f"  点数: {scene.num_points}")
    
    # [2] モック画像作成
    print("\n[2/6] モック画像作成中...")
    image_dir = output_dir / 'images'
    image_dir.mkdir(parents=True, exist_ok=True)
    
    image_paths = []
    for i in range(num_views):
        img = Image.new('RGB', (512, 512), color=(128, 128, 128))
        img_path = image_dir / f'view_{i:02d}.jpg'
        img.save(img_path)
        image_paths.append(str(img_path))
    
    print(f"  ✓ {len(image_paths)}枚の画像作成完了")
    
    # [3] Process2でパラメータ抽出
    print("\n[3/6] Process2でカメラパラメータ抽出中...")
    cameras_dict, pts3d, confidence = extract_camera_params_process2(
        scene, image_paths, conf_threshold=conf_threshold
    )
    
    print(f"\n抽出結果:")
    print(f"  pts3d shape: {pts3d.shape}")
    print(f"  confidence shape: {confidence.shape}")
    print(f"  cameras: {len(cameras_dict)}")
    
    # [4] COLMAP形式で保存 (バイナリ)
    print("\n[4/6] COLMAP形式(バイナリ)で保存中...")
    img = Image.open(image_paths[0])
    image_size = img.size
    img.close()
    
    sparse_dir = export_colmap_binary(
        cameras_dict, pts3d, confidence, image_size, output_dir / 'sparse_process2' / '0'
    )
    
    # [5] テキスト形式でも保存 (検証用)
    print("\n[5/6] テキスト形式で保存中...")
    text_dir = save_colmap_text_process2(
        cameras_dict, pts3d, confidence, output_dir
    )
    
    # [6] 検証
    print("\n[6/6] 座標変換検証中...")
    success = verify_transformation(ground_truth_dir, text_dir)
    
    print("\n" + "="*70)
    if success:
        print("✓ テスト完了!")
    else:
        print("⚠ テスト完了 (警告あり)")
    print(f"結果: {output_dir}")
    print(f"  Ground Truth: {ground_truth_dir}")
    print(f"  COLMAP Binary: {sparse_dir}")
    print(f"  COLMAP Text: {text_dir}")
    print("="*70)
    
    return scene, cameras_dict, pts3d, confidence


# ============================================================================
# 使用例
# ============================================================================

if __name__ == "__main__":
    print("="*70)
    print("Process2 COLMAP Method + Calibration Test")
    print("="*70)
    print("\n使用方法:")
    print("  scene, cameras_dict, pts3d, confidence = test_process2_with_calibration()")
    print("\nこれにより:")
    print("  1. キャリブレーションデータ生成")
    print("  2. Process2で処理")
    print("  3. COLMAP形式で出力")
    print("  4. 座標変換の検証")
    print("が全て自動で実行されます。")

Process2 COLMAP Method + Calibration Test

使用方法:
  scene, cameras_dict, pts3d, confidence = test_process2_with_calibration()

これにより:
  1. キャリブレーションデータ生成
  2. Process2で処理
  3. COLMAP形式で出力
  4. 座標変換の検証
が全て自動で実行されます。


In [4]:
scene, cameras_dict, pts3d, confidence = test_process2_with_calibration(num_views=4, num_points=100, 
                                   output_dir='/kaggle/working/calibration_test_process2',
                                   conf_threshold=1.5)

Process2 COLMAP Method + Calibration Test

[1/6] キャリブレーションデータ生成中...
✓ Ground truth saved to /kaggle/working/calibration_test_process2/ground_truth
  ビュー数: 4
  点数: 92

[2/6] モック画像作成中...
  ✓ 4枚の画像作成完了

[3/6] Process2でカメラパラメータ抽出中...

=== Extracting Camera Parameters ===

Example camera 0:
  Original size: 512x512
  MASt3R size: 224.0
  Scale factor: 2.286
  focals.shape: torch.Size([4, 1])
  MASt3R focal: 500.00
  Scaled focal: fx = fy = 1142.86
  MASt3R pp: [112.00, 112.00]
  Scaled pp: [256.00, 256.00]
✓ Extracted parameters for 4 cameras
✓ Total 3D points: 368
✓ Points after confidence filtering (>1.5): 368

抽出結果:
  pts3d shape: (368, 3)
  confidence shape: (368,)
  cameras: 4

[4/6] COLMAP形式(バイナリ)で保存中...
COLMAP cameras.bin saved to /kaggle/working/calibration_test_process2/sparse_process2/0/cameras.bin
COLMAP images.bin saved to /kaggle/working/calibration_test_process2/sparse_process2/0/images.bin
COLMAP points3D.bin saved to /kaggle/working/calibration_test_process2/sparse_process2/